In [ ]:
## uncomment this if you are testing a 
## non-installed version of the Python API and update the path
##
#import sys
#sys.path.insert(0, r"C:\SVN\achapkowski_geosaurus_fork\src")

In [ ]:
from arcgis.gis import GIS, ContentManager, Item
import pandas as pd
import pytest
import os
from utils import NOTEBOOK_TESTS_DIR
nb_dir = os.path.join(NOTEBOOK_TESTS_DIR, "common_workflows")

In [ ]:
profiles = ['your_online_profile', 'your_enterprise_profile']
gis_objects = {}
for profile in profiles:
    gis = GIS(profile=profile, verify_cert=False)
    gis_objects[profile] = gis

### Content Manager Methods

In [ ]:
from datetime import datetime
for profile in profiles:
    gis = gis_objects[profile]
    cm = gis.content
    assert isinstance(cm, ContentManager)
    assert len(cm.advanced_search("orgid: %s" % gis.properties.id, max_items=500)['results']) >= 0
    assert len(gis.content.search("orgid: %s" % gis.properties.id, max_items=500)) >= 0
    item = gis.content.search("orgid: %s & owner: %s" % (gis.properties.id,
                                             gis.users.me.username), max_items=1)[0]
    assert cm.can_delete(item)['success'] in (True, False)
    assert cm.categories
    folder = "eraseme" + str(datetime.now().microsecond)[:3]
    assert 'title' in cm.create_folder(folder)
    assert cm.delete_folder(folder)
    assert cm.is_service_name_available(folder, "featureService") in (True, False)
    with open(os.path.join(nb_dir,"webmap.json"), 'r') as reader:
        text = reader.read()
        add_item = cm.add(
            item_properties={'title' : folder,
                             'text' : text,
                             'type' : 'Web Map', 'tags' : 'erase, me'},
            data=None)
        assert isinstance(add_item, Item)
        assert add_item.delete()
    sd_file = os.path.join(nb_dir, "test002.sd")
    add_item = cm.add(
            item_properties={'title' : folder, 'type' : 'Service Definition', 'tags' : 'erase, me'},
            data=sd_file
        )
    assert isinstance(add_item, Item)
    assert add_item.delete()

### Publishing Test

In [ ]:
from datetime import datetime
for profile in profiles:
    gis = gis_objects[profile]
    folder = "eraseme" + str(datetime.now().microsecond)[:3]
    cm = gis.content
    sd_file = os.path.join(nb_dir, "test002.sd")
    add_item = cm.add(
        item_properties={'title' : folder, 'type' : 'Service Definition', 'tags' : 'erase, me'},
        data=sd_file)
    assert isinstance(add_item, Item)
    result = add_item.publish()
    assert isinstance(result, Item)
    assert result.delete()
    assert add_item.delete()